In [ ]:
!pip install -q google-genai

In [ ]:
import json, time
from google.colab import userdata
from google import genai
from google.genai import types

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
gclient = genai.Client(api_key=GEMINI_API_KEY)
JUDGE_MODEL = "gemini-3-flash-preview"

In [ ]:
FILE_PATH = 'RPT_AUGMENTED_150.json'

with open(FILE_PATH, encoding='utf-8') as f:
    dataset = json.load(f)

print(f"검증 대상: {len(dataset)}개")

In [ ]:
import json
import time

JUDGE_DEFINITIONS = """[평가기준 정의]

1. 완전성(COMP)
- 개념: 계획(PEP)·RFP의 과업·업무범위 항목이 결과보고서(RPT)에 내용상 빠짐없이 이행·보고됐는가? (필수 섹션·필드의 단순 존재 여부는 판단 대상 아님)
- 주의사항: LLM은 "PEP에 계획된 과업/범위 항목이 실제로 이행되고 그 내용이 RPT에 서술됐는가"만 판단한다. 사업명·사업기간·사업예산 등 단일 필수 필드가 문서에 존재하는지 여부는 룰베이스(정규식/체크리스트) 사전점검 대상이며, 완전성 판단 근거에 포함하지 않는다. 필수 섹션·산출물 항목의 존재 여부도 룰베이스 대상이다.
- 충족: PEP에 열거된 과업·범위 항목이 RPT에 모두 대응되어 확인됨 (표현이 다르더라도 의미상 동일 항목이면 인정)
- 불가: PEP에 열거된 과업·범위 항목 중 하나 이상이 RPT 어디에도(유사 표현 포함) 확인되지 않음. 판정 시 반드시 ① PEP 원문 목록 전체, ② RPT에서 실제로 대응되는 항목, ③ 대응되지 않는 항목을 각각 명시하고, 단순 표현 차이가 아니라 실질적으로 부재함을 근거로 제시한다. (동의어·패러프레이즈로 다른 곳에 언급되지 않았는지 확인한 흔적을 코멘트에 남긴다)
- 검토: 
  ① 대조할 PEP/RFP 자체가 미첨부됨
  ② 요구사항이 조건부 문구(~하는 경우)라 해당 항목의 필수 여부가 맥락에 따라 달라져 판단이 애매함

2. 정확성(ACC)
- 개념: 보고된 사실·규격·수치가 정확하고 계획·실제와 모순이 없는가?
- 충족: 수치·규격이 정확하며 계획·계약과 일치함
- 불가: 명백한 부정확·모순 (예: 버전 오기 / 문서 내 기간·금액 불일치 — 내부 모순은 대조 자료가 없어도 불가 판정 가능)
- 검토: 외부 수치 대조 불가 (계획·계약 미첨부)

3. 검증가능성(VER)
- 개념: 성과·결과가 측정·확인 가능하게(정량·구체) 보고됐는가?
- 충족: 정량 성과·검증 기준이 명확함 (예: "잔존 심각결함 0건", "가동률 99%")
- 불가: "적절히/최선을 다해" 등 모호한 표현뿐인 경우
- 검토: 정량·모호 표현이 혼재된 경우

4. 추적성(TRACE)
- 개념: 계획 과업 → Task → 산출물 → 실제 결과가 연결·추적되는가?
- 충족: 과업-Task-산출물-결과가 1:1로 연결됨
- 불가: 연결이 단절된 경우
  ① 산출물은 제출됐는데 대응 과업·결과가 없음
  ② 계획 단계·과업·산출물 명칭을 결과보고서가 임의로 재구성하여 계획-결과 간 1:1 대응을 확인할 수 없음
- 검토: 
  ① 부분 연결
  ② 확인 대상 섹션이 입력에 없음
"""

JUDGE_PROMPT = f"""당신은 결과보고서(RPT) 평가 판정을 독립적으로 내리는 평가자입니다.
아래 [평가기준 정의]를 근거로, 주어진 target_criteria에 대해 PEP_excerpt와 RPT_excerpt를 직접 대조하여 판정하십시오.
다른 사람이 이미 내린 판정은 제공되지 않으며, 오직 이 기준에만 의존하여 엄격하게 스스로 판단해야 합니다.

{JUDGE_DEFINITIONS}

target_description.quality에 있는 기준 중 target_criteria로 지정된 것만 판단하십시오.
target_criteria가 2개 이상이면 기준별로 각각 충족/불가/검토를 판단한 뒤, 종합판정 규칙에 따라 overall_verdict를 결정하십시오.
PEP_mapping이 비어 있는 절이면 RPT 자체의 검증가능성만 판단하십시오.

반드시 아래 JSON 형식으로만 답하십시오.
{{
  "criteria_verdicts": {{"완전성": "충족|불가|검토", "정확성": "충족|불가|검토", ...}},
  "overall_verdict": "충족" 또는 "불가" 또는 "검토",
  "reasoning": "기준별 판단 근거를 한두 문장씩 작성"
}}
"""

def judge_item(item, retries=3):
    desc = item['input']['description']
    payload = {
        "target_description": {
            "id": desc.get("id"),
            "chapter": desc.get("chapter"),
            "section": desc.get("section"),
            "standard_structure": desc.get("standard_structure"),
        },
        "target_criteria": desc.get("quality", []),
        "PEP_excerpt": item['input'].get('PEP_excerpt', ''),
        "RPT_excerpt": item['input'].get('RPT_excerpt', ''),
    }
    user_msg = json.dumps(payload, ensure_ascii=False)

    for attempt in range(retries):
        try:
            resp = gclient.models.generate_content(
                model=JUDGE_MODEL,
                contents=f"{JUDGE_PROMPT}\n\n판정 대상:\n{user_msg}",
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                ),
            )
            return json.loads(resp.text)
        except Exception as e:
            print(f"  [warn] judge 호출 실패(attempt {attempt+1}): {e}")
            time.sleep(5)
    return {"overall_verdict": "ERROR", "criteria_verdicts": {}, "reasoning": "judge 호출 실패"}

In [ ]:
results = []
mismatches = []

for i, item in enumerate(dataset):
    original_verdict = item['output']['판정']
    gemini = judge_item(item)
    predicted_verdict = gemini.get('overall_verdict')

    agree = (predicted_verdict == original_verdict)
    results.append({
        "id": item.get("id"),
        "gpt_판정": original_verdict,
        "gemini_판정": predicted_verdict,
        "agree": agree,
        "criteria_verdicts": gemini.get("criteria_verdicts"),
        "reasoning": gemini.get("reasoning"),
    })

    tag = "일치" if agree else "불일치"
    print(f"[{i+1}/{len(dataset)}] {item.get('id')} | GPT={original_verdict} vs Gemini={predicted_verdict} -> {tag}")

    if not agree:
        mismatches.append(item.get("id"))

    time.sleep(7)

    with open('agreement_results_pep.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

agreement_rate = sum(r['agree'] for r in results) / len(results)
print(f"\n총 {len(dataset)}개 중 일치 {sum(r['agree'] for r in results)}개 ({agreement_rate:.1%})")
print("불일치 id 목록:", mismatches)

In [ ]:
from collections import Counter
confusion = Counter((r['gpt_판정'], r['gemini_판정']) for r in results)
print("혼동행렬 (GPT라벨, Gemini라벨) -> 개수")
for k, v in sorted(confusion.items(), key=lambda x: -x[1]):
    marker = "  <- 불일치" if k[0] != k[1] else ""
    print(f"  {k}: {v}{marker}")